In [0]:
# This block MUST run first to write the mock data files!
dbutils.fs.put("dbfs:/tmp/doctors.csv", """doctor_id,doctor_name,specialization,city,experience_years,consultation_fee
D101,Dr. Ramesh,Cardiology,Hyderabad,12,1500
...""", overwrite=True)

dbutils.fs.put("dbfs:/tmp/visits.csv", """visit_id,patient_name,doctor_id,visit_date,diagnosis,bill_amount,payment_status
V1001,Rahul Sharma,D101,2026-01-10,Heart Checkup,5000,Paid
...""", overwrite=True)

Wrote 124 bytes.
Wrote 142 bytes.


True

In [0]:
df_docs = spark.read.option("header", "true").option("inferSchema", "true").csv("dbfs:/tmp/doctors.csv")
df_visits = spark.read.option("header", "true").option("inferSchema", "true").csv("dbfs:/tmp/visits.csv")

In [0]:
df_docs.printSchema()
df_visits.printSchema()

root
 |-- doctor_id: string (nullable = true)
 |-- doctor_name: string (nullable = true)
 |-- specialization: string (nullable = true)
 |-- city: string (nullable = true)
 |-- experience_years: integer (nullable = true)
 |-- consultation_fee: integer (nullable = true)

root
 |-- visit_id: string (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- doctor_id: string (nullable = true)
 |-- visit_date: date (nullable = true)
 |-- diagnosis: string (nullable = true)
 |-- bill_amount: integer (nullable = true)
 |-- payment_status: string (nullable = true)



In [0]:
print(f"Total Doctors: {df_docs.count()} | Total Visits: {df_visits.count()}")

Total Doctors: 8 | Total Visits: 10


In [0]:
df_docs.filter(F.col("city") == "Hyderabad").show()

+---------+-----------+--------------+---------+----------------+----------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_fee|
+---------+-----------+--------------+---------+----------------+----------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|            1500|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|              20|            3000|
+---------+-----------+--------------+---------+----------------+----------------+



In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
dbutils.fs.put("dbfs:/tmp/doctors.csv", """doctor_id,doctor_name,specialization,city,experience_years,consultation_fee
D101,Dr. Ramesh,Cardiology,Hyderabad,12,1500
D102,Dr. Priya,Neurology,Bangalore,10,2000
D103,Dr. Anita,Dermatology,Chennai,8,1000
D104,Dr. Suresh,Orthopedics,Mumbai,15,2500
D105,Dr. Meera,Pediatrics,Delhi,6,1200
D106,Dr. Kiran,Cardiology,Hyderabad,20,3000
D107,Dr. Farhan,Neurology,Pune,5,1800
D108,Dr. Nisha,Dermatology,Kochi,9,1100""", overwrite=True)

Wrote 409 bytes.


True

In [0]:
dbutils.fs.put("dbfs:/tmp/visits.csv", """visit_id,patient_name,doctor_id,visit_date,diagnosis,bill_amount,payment_status
V1001,Rahul Sharma,D101,2026-01-10,Heart Checkup,5000,Paid
V1002,Priya Reddy,D102,2026-01-10,Migraine,3500,Paid
V1003,Amit Kumar,D103,2026-01-11,Skin Allergy,2000,Pending
V1004,Sneha Patel,D104,2026-01-11,Fracture,12000,Paid
V1005,Farhan Ali,D105,2026-01-12,Fever,1500,Paid
V1006,Neha Singh,D106,2026-01-12,Heart Checkup,7000,Paid
V1007,Arjun Verma,D120,2026-01-13,Migraine,4000,Paid
V1008,Meera Nair,D103,2026-01-13,Skin Allergy,,Pending
V1009,Kiran Rao,D104,2026-01-14,Back Pain,6000,Paid
V1010,Nisha Reddy,D101,2026-01-14,Heart Checkup,5500,Paid""", overwrite=True)

Wrote 628 bytes.


True

In [0]:
df_docs = spark.read.option("header", "true").csv("dbfs:/tmp/doctors.csv")
df_docs.filter(F.col("specialization") == "Cardiology").show()


+---------+-----------+--------------+---------+----------------+----------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_fee|
+---------+-----------+--------------+---------+----------------+----------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|            1500|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|              20|            3000|
+---------+-----------+--------------+---------+----------------+----------------+



In [0]:
df_docs = spark.read.csv('dbfs:/tmp/doctors.csv', header=True, inferSchema=True)
df_docs.filter(F.col("experience_years") > 10).show()



+---------+-----------+--------------+---------+----------------+----------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_fee|
+---------+-----------+--------------+---------+----------------+----------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|            1500|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|              15|            2500|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|              20|            3000|
+---------+-----------+--------------+---------+----------------+----------------+



In [0]:
df_docs.groupBy("specialization").agg(F.avg("consultation_fee").alias("avg_fee")).show()

+--------------+-------+
|specialization|avg_fee|
+--------------+-------+
|     Neurology| 1900.0|
|   Dermatology| 1050.0|
|    Cardiology| 2250.0|
|    Pediatrics| 1200.0|
|   Orthopedics| 2500.0|
+--------------+-------+



In [0]:
df_docs.groupBy("specialization").agg(F.max("consultation_fee").alias("max_fee")).show()

+--------------+-------+
|specialization|max_fee|
+--------------+-------+
|     Neurology|   2000|
|   Dermatology|   1100|
|    Cardiology|   3000|
|    Pediatrics|   1200|
|   Orthopedics|   2500|
+--------------+-------+



In [0]:
df_docs.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    1|
|    Kochi|    1|
|  Chennai|    1|
|   Mumbai|    1|
|     Pune|    1|
|    Delhi|    1|
|Hyderabad|    2|
+---------+-----+



In [0]:
df_docs.groupBy("specialization").count().show()

+--------------+-----+
|specialization|count|
+--------------+-----+
|     Neurology|    2|
|   Dermatology|    2|
|    Cardiology|    2|
|    Pediatrics|    1|
|   Orthopedics|    1|
+--------------+-----+



In [0]:
df_visits.select(F.sum("bill_amount").alias("total_collection")).show()

+----------------+
|total_collection|
+----------------+
|            5000|
+----------------+



In [0]:
df_visits.select(F.avg("bill_amount").alias("avg_bill")).show()

+--------+
|avg_bill|
+--------+
|  5000.0|
+--------+



In [0]:
df_visits.select(F.max("bill_amount"), F.min("bill_amount")).show()

+----------------+----------------+
|max(bill_amount)|min(bill_amount)|
+----------------+----------------+
|            5000|            5000|
+----------------+----------------+



In [0]:
df_docs.orderBy(F.col("consultation_fee").desc()).show()

+---------+-----------+--------------+---------+----------------+----------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_fee|
+---------+-----------+--------------+---------+----------------+----------------+
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|              20|            3000|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|              15|            2500|
|     D102|  Dr. Priya|     Neurology|Bangalore|              10|            2000|
|     D107| Dr. Farhan|     Neurology|     Pune|               5|            1800|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|            1500|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|               6|            1200|
|     D108|  Dr. Nisha|   Dermatology|    Kochi|               9|            1100|
|     D103|  Dr. Anita|   Dermatology|  Chennai|               8|            1000|
+---------+-----------+--------------+---------+----------------+----------------+



In [0]:
df_visits.orderBy(F.col("bill_amount").desc()).show()

+--------+------------+---------+----------+-------------+-----------+--------------+
|visit_id|patient_name|doctor_id|visit_date|    diagnosis|bill_amount|payment_status|
+--------+------------+---------+----------+-------------+-----------+--------------+
|   V1001|Rahul Sharma|     D101|2026-01-10|Heart Checkup|       5000|          Paid|
|     ...|        NULL|     NULL|      NULL|         NULL|       NULL|          NULL|
+--------+------------+---------+----------+-------------+-----------+--------------+



In [0]:
df_visits.filter(F.col("bill_amount").isNull()).show()

+--------+------------+---------+----------+---------+-----------+--------------+
|visit_id|patient_name|doctor_id|visit_date|diagnosis|bill_amount|payment_status|
+--------+------------+---------+----------+---------+-----------+--------------+
|     ...|        NULL|     NULL|      NULL|     NULL|       NULL|          NULL|
+--------+------------+---------+----------+---------+-----------+--------------+



In [0]:
df_visits_clean = df_visits.fillna({"bill_amount": 0})
df_visits_clean.show()

+--------+------------+---------+----------+-------------+-----------+--------------+
|visit_id|patient_name|doctor_id|visit_date|    diagnosis|bill_amount|payment_status|
+--------+------------+---------+----------+-------------+-----------+--------------+
|   V1001|Rahul Sharma|     D101|2026-01-10|Heart Checkup|       5000|          Paid|
|     ...|        NULL|     NULL|      NULL|         NULL|          0|          NULL|
+--------+------------+---------+----------+-------------+-----------+--------------+



In [0]:
df_visits_tax = df_visits_clean.withColumn("tax", F.col("bill_amount") * 0.05)

In [0]:
df_visits_final = df_visits_tax.withColumn("final_bill", F.col("bill_amount") + F.col("tax"))
df_visits_final.show()

+--------+------------+---------+----------+-------------+-----------+--------------+-----+----------+
|visit_id|patient_name|doctor_id|visit_date|    diagnosis|bill_amount|payment_status|  tax|final_bill|
+--------+------------+---------+----------+-------------+-----------+--------------+-----+----------+
|   V1001|Rahul Sharma|     D101|2026-01-10|Heart Checkup|       5000|          Paid|250.0|    5250.0|
|     ...|        NULL|     NULL|      NULL|         NULL|          0|          NULL|  0.0|       0.0|
+--------+------------+---------+----------+-------------+-----------+--------------+-----+----------+



In [0]:
df_inner = df_docs.join(df_visits_clean, "doctor_id", "inner")
df_inner.show()

+---------+-----------+--------------+---------+----------------+----------------+--------+------------+----------+-------------+-----------+--------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_fee|visit_id|patient_name|visit_date|    diagnosis|bill_amount|payment_status|
+---------+-----------+--------------+---------+----------------+----------------+--------+------------+----------+-------------+-----------+--------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|            1500|   V1001|Rahul Sharma|2026-01-10|Heart Checkup|       5000|          Paid|
+---------+-----------+--------------+---------+----------------+----------------+--------+------------+----------+-------------+-----------+--------------+



In [0]:
df_left = df_docs.join(df_visits_clean, "doctor_id", "left")
df_left.show()

+---------+-----------+--------------+---------+----------------+----------------+--------+------------+----------+-------------+-----------+--------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_fee|visit_id|patient_name|visit_date|    diagnosis|bill_amount|payment_status|
+---------+-----------+--------------+---------+----------------+----------------+--------+------------+----------+-------------+-----------+--------------+
|      ...|       NULL|          NULL|     NULL|            NULL|            NULL|    NULL|        NULL|      NULL|         NULL|       NULL|          NULL|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|            1500|   V1001|Rahul Sharma|2026-01-10|Heart Checkup|       5000|          Paid|
+---------+-----------+--------------+---------+----------------+----------------+--------+------------+----------+-------------+-----------+--------------+



In [0]:
df_right = df_docs.join(df_visits_clean, "doctor_id", "right")
df_right.show()

+---------+-----------+--------------+---------+----------------+----------------+--------+------------+----------+-------------+-----------+--------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_fee|visit_id|patient_name|visit_date|    diagnosis|bill_amount|payment_status|
+---------+-----------+--------------+---------+----------------+----------------+--------+------------+----------+-------------+-----------+--------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|            1500|   V1001|Rahul Sharma|2026-01-10|Heart Checkup|       5000|          Paid|
|     NULL|       NULL|          NULL|     NULL|            NULL|            NULL|     ...|        NULL|      NULL|         NULL|          0|          NULL|
+---------+-----------+--------------+---------+----------------+----------------+--------+------------+----------+-------------+-----------+--------------+



In [0]:

df_full = df_docs.join(df_visits_clean, "doctor_id", "full")
df_full.show()

+---------+-----------+--------------+---------+----------------+----------------+--------+------------+----------+-------------+-----------+--------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_fee|visit_id|patient_name|visit_date|    diagnosis|bill_amount|payment_status|
+---------+-----------+--------------+---------+----------------+----------------+--------+------------+----------+-------------+-----------+--------------+
|     NULL|       NULL|          NULL|     NULL|            NULL|            NULL|     ...|        NULL|      NULL|         NULL|          0|          NULL|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|            1500|   V1001|Rahul Sharma|2026-01-10|Heart Checkup|       5000|          Paid|
|      ...|       NULL|          NULL|     NULL|            NULL|            NULL|    NULL|        NULL|      NULL|         NULL|       NULL|          NULL|
+---------+-----------+--------------+---------+----------

In [0]:
df_visits_clean.join(df_docs, "doctor_id", "left_anti").show()

+---------+--------+------------+----------+---------+-----------+--------------+
|doctor_id|visit_id|patient_name|visit_date|diagnosis|bill_amount|payment_status|
+---------+--------+------------+----------+---------+-----------+--------------+
|     NULL|     ...|        NULL|      NULL|     NULL|          0|          NULL|
+---------+--------+------------+----------+---------+-----------+--------------+



In [0]:
df_docs.join(df_visits_clean, "doctor_id", "left_anti").show()

+---------+-----------+--------------+----+----------------+----------------+
|doctor_id|doctor_name|specialization|city|experience_years|consultation_fee|
+---------+-----------+--------------+----+----------------+----------------+
|      ...|       NULL|          NULL|NULL|            NULL|            NULL|
+---------+-----------+--------------+----+----------------+----------------+



In [0]:
df_left.groupBy("doctor_id", "doctor_name").agg(F.count("visit_id").alias("visit_count")).show()

+---------+-----------+-----------+
|doctor_id|doctor_name|visit_count|
+---------+-----------+-----------+
|      ...|       NULL|          0|
|     D101| Dr. Ramesh|          1|
+---------+-----------+-----------+



In [0]:
df_doc_rev = df_left.groupBy("doctor_id", "doctor_name").agg(F.sum("bill_amount").alias("doctor_revenue")).fillna(0)
df_doc_rev.show()

+---------+-----------+--------------+
|doctor_id|doctor_name|doctor_revenue|
+---------+-----------+--------------+
|      ...|       NULL|             0|
|     D101| Dr. Ramesh|          5000|
+---------+-----------+--------------+



In [0]:
df_doc_rev.orderBy(F.col("doctor_revenue").desc()).limit(1).show()

+---------+-----------+--------------+
|doctor_id|doctor_name|doctor_revenue|
+---------+-----------+--------------+
|     D101| Dr. Ramesh|          5000|
+---------+-----------+--------------+



In [0]:
df_left.groupBy("specialization").agg(F.sum("bill_amount").alias("spec_revenue")).orderBy(F.desc("spec_revenue")).limit(1).show()

+--------------+------------+
|specialization|spec_revenue|
+--------------+------------+
|    Cardiology|        5000|
+--------------+------------+



In [0]:
df_left.groupBy("specialization").agg(F.avg("bill_amount").alias("avg_spec_revenue")).show()

+--------------+----------------+
|specialization|avg_spec_revenue|
+--------------+----------------+
|          NULL|            NULL|
|    Cardiology|          5000.0|
+--------------+----------------+



In [0]:
df_left.groupBy("city").agg(F.sum("bill_amount").alias("city_revenue")).show()

+---------+------------+
|     city|city_revenue|
+---------+------------+
|     NULL|        NULL|
|Hyderabad|        5000|
+---------+------------+



In [0]:
df_left.groupBy("doctor_name").count().show()

+-----------+-----+
|doctor_name|count|
+-----------+-----+
|       NULL|    1|
| Dr. Ramesh|    1|
+-----------+-----+



In [0]:
df_doc_rev.orderBy(F.desc("doctor_revenue")).limit(3).show()

+---------+-----------+--------------+
|doctor_id|doctor_name|doctor_revenue|
+---------+-----------+--------------+
|     D101| Dr. Ramesh|          5000|
|      ...|       NULL|             0|
+---------+-----------+--------------+



In [0]:
df_left.groupBy("doctor_id", "doctor_name", "specialization") \
       .agg(F.count("visit_id").alias("total_patients"), 
            F.sum("bill_amount").alias("total_revenue"),
            F.avg("bill_amount").alias("avg_bill_per_patient")) \
       .fillna(0).show()

+---------+-----------+--------------+--------------+-------------+--------------------+
|doctor_id|doctor_name|specialization|total_patients|total_revenue|avg_bill_per_patient|
+---------+-----------+--------------+--------------+-------------+--------------------+
|      ...|       NULL|          NULL|             0|            0|                 0.0|
|     D101| Dr. Ramesh|    Cardiology|             1|         5000|              5000.0|
+---------+-----------+--------------+--------------+-------------+--------------------+



In [0]:
dbutils.fs.put("dbfs:/tmp/hospital_config.json", """[
{"hospital_id":"H101","hospital_name":"Apollo Hospital","city":"Hyderabad","contact":{"phone":"9876500011","email":"apollo@mail.com"},"services":["Cardiology","Neurology","Dermatology"]},
{"hospital_id":"H102","hospital_name":"Yashoda Hospital","city":"Hyderabad","contact":{"phone":null,"email":"yashoda@mail.com"},"services":["Cardiology","Orthopedics"]},
{"hospital_id":"H103","hospital_name":"Care Hospital","city":"Bangalore","contact":{"phone":"9876500013","email":null},"services":["Neurology","Pediatrics"]}
]""", overwrite=True)

df_json = spark.read.option("multiline", "true").json("dbfs:/tmp/hospital_config.json")

Wrote 519 bytes.


In [0]:
df_json.printSchema()

root
 |-- city: string (nullable = true)
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- hospital_id: string (nullable = true)
 |-- hospital_name: string (nullable = true)
 |-- services: array (nullable = true)
 |    |-- element: string (containsNull = true)



In [0]:
df_flat_contact = df_json.withColumn("phone", F.col("contact.phone")).withColumn("email", F.col("contact.email"))
df_flat_contact.select("hospital_name", "phone", "email").show()

+----------------+----------+----------------+
|   hospital_name|     phone|           email|
+----------------+----------+----------------+
| Apollo Hospital|9876500011| apollo@mail.com|
|Yashoda Hospital|      NULL|yashoda@mail.com|
|   Care Hospital|9876500013|            NULL|
+----------------+----------+----------------+



In [0]:
df_flat_contact.filter(F.col("phone").isNull()).select("hospital_name").show()

+----------------+
|   hospital_name|
+----------------+
|Yashoda Hospital|
+----------------+



In [0]:
df_flat_contact.filter(F.col("email").isNull()).select("hospital_name").show()

+-------------+
|hospital_name|
+-------------+
|Care Hospital|
+-------------+



In [0]:
df_json_filled = df_flat_contact.fillna({"phone": "N/A", "email": "missing@hospital.com"})

In [0]:
df_json.select("hospital_name", "city").show()

+----------------+---------+
|   hospital_name|     city|
+----------------+---------+
| Apollo Hospital|Hyderabad|
|Yashoda Hospital|Hyderabad|
|   Care Hospital|Bangalore|
+----------------+---------+



In [0]:
df_flat_contact.select("hospital_name", "phone").show()

+----------------+----------+
|   hospital_name|     phone|
+----------------+----------+
| Apollo Hospital|9876500011|
|Yashoda Hospital|      NULL|
|   Care Hospital|9876500013|
+----------------+----------+



In [0]:
df_exploded = df_json_filled.withColumn("service", F.explode(F.col("services")))
df_exploded.select("hospital_name", "service").show()

+----------------+-----------+
|   hospital_name|    service|
+----------------+-----------+
| Apollo Hospital| Cardiology|
| Apollo Hospital|  Neurology|
| Apollo Hospital|Dermatology|
|Yashoda Hospital| Cardiology|
|Yashoda Hospital|Orthopedics|
|   Care Hospital|  Neurology|
|   Care Hospital| Pediatrics|
+----------------+-----------+



In [0]:
df_json.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    1|
|Hyderabad|    2|
+---------+-----+



In [0]:
df_exploded.groupBy("service").count().show()

+-----------+-----+
|    service|count|
+-----------+-----+
|  Neurology|    2|
|Dermatology|    1|
| Cardiology|    2|
| Pediatrics|    1|
|Orthopedics|    1|
+-----------+-----+



In [0]:
print("Cardiology:")
df_json.filter(F.array_contains(F.col("services"), "Cardiology")).select("hospital_name").show()
print("Neurology:")
df_json.filter(F.array_contains(F.col("services"), "Neurology")).select("hospital_name").show()
print("Orthopedics:")
df_json.filter(F.array_contains(F.col("services"), "Orthopedics")).select("hospital_name").show()
print("Pediatrics:")
df_json.filter(F.array_contains(F.col("services"), "Pediatrics")).select("hospital_name").show()

Cardiology:
+----------------+
|   hospital_name|
+----------------+
| Apollo Hospital|
|Yashoda Hospital|
+----------------+

Neurology:
+---------------+
|  hospital_name|
+---------------+
|Apollo Hospital|
|  Care Hospital|
+---------------+

Orthopedics:
+----------------+
|   hospital_name|
+----------------+
|Yashoda Hospital|
+----------------+

Pediatrics:
+-------------+
|hospital_name|
+-------------+
|Care Hospital|
+-------------+



In [0]:
df_exploded.drop("contact", "services").write.mode("overwrite").parquet("/tmp/outputs/hospitals_parquet")

In [0]:
df_exploded.drop("contact", "services").write.mode("overwrite").option("header","true").csv("/tmp/outputs/hospitals_csv")

In [0]:
df_rev_base = df_left.groupBy("doctor_id", "doctor_name", "specialization", "city") \
                     .agg(F.sum("bill_amount").alias("revenue")).fillna(0)

In [0]:
win_global_rev = Window.orderBy(F.desc("revenue"))
win_spec_rev = Window.partitionBy("specialization").orderBy(F.desc("revenue"))
win_city_rev = Window.partitionBy("city").orderBy(F.desc("revenue"))
win_running = Window.orderBy("doctor_id").rowsBetween(Window.unboundedPreceding, Window.currentRow)

In [0]:
df_win_metric = df_rev_base.withColumn("rank", F.rank().over(win_global_rev))

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
df_win_metric = df_win_metric.withColumn("dense_rank", F.dense_rank().over(win_global_rev))

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
df_win_metric = df_win_metric.withColumn("row_number", F.row_number().over(win_global_rev))
df_win_metric.show()

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+----+----------+----------+
|doctor_id|doctor_name|specialization|     city|revenue|rank|dense_rank|row_number|
+---------+-----------+--------------+---------+-------+----+----------+----------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|   5000|   1|         1|         1|
|      ...|       NULL|          NULL|     NULL|      0|   2|         2|         2|
+---------+-----------+--------------+---------+-------+----+----------+----------+



In [0]:
df_win_metric.filter(F.col("row_number") <= 3).show()

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+----+----------+----------+
|doctor_id|doctor_name|specialization|     city|revenue|rank|dense_rank|row_number|
+---------+-----------+--------------+---------+-------+----+----------+----------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|   5000|   1|         1|         1|
|      ...|       NULL|          NULL|     NULL|      0|   2|         2|         2|
+---------+-----------+--------------+---------+-------+----+----------+----------+



In [0]:
df_rev_base.withColumn("spec_rank", F.row_number().over(win_spec_rev)).filter(F.col("spec_rank") <= 2).show()

+---------+-----------+--------------+---------+-------+---------+
|doctor_id|doctor_name|specialization|     city|revenue|spec_rank|
+---------+-----------+--------------+---------+-------+---------+
|      ...|       NULL|          NULL|     NULL|      0|        1|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|   5000|        1|
+---------+-----------+--------------+---------+-------+---------+



In [0]:
df_rev_base.withColumn("running_total", F.sum("revenue").over(win_running)).show()

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+-------------+
|doctor_id|doctor_name|specialization|     city|revenue|running_total|
+---------+-----------+--------------+---------+-------+-------------+
|      ...|       NULL|          NULL|     NULL|      0|            0|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|   5000|         5000|
+---------+-----------+--------------+---------+-------+-------------+



In [0]:
df_lag_lead = df_rev_base.withColumn("prev_doc_rev", F.lag("revenue", 1).over(win_global_rev)) \
                         .withColumn("next_doc_rev", F.lead("revenue", 1).over(win_global_rev))

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:

df_lag_lead.select("doctor_name", "revenue", "prev_doc_rev", "next_doc_rev").show()

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+-------+------------+------------+
|doctor_name|revenue|prev_doc_rev|next_doc_rev|
+-----------+-------+------------+------------+
| Dr. Ramesh|   5000|        NULL|           0|
|       NULL|      0|        5000|        NULL|
+-----------+-------+------------+------------+



In [0]:
df_rev_base.withColumn("city_rank", F.row_number().over(win_city_rev)).filter(F.col("city_rank") == 1).show()

+---------+-----------+--------------+---------+-------+---------+
|doctor_id|doctor_name|specialization|     city|revenue|city_rank|
+---------+-----------+--------------+---------+-------+---------+
|      ...|       NULL|          NULL|     NULL|      0|        1|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|   5000|        1|
+---------+-----------+--------------+---------+-------+---------+



In [0]:
win_city_rev_asc = Window.partitionBy("city").orderBy(F.col("revenue").asc())
df_rev_base.withColumn("city_rank_lowest", F.row_number().over(win_city_rev_asc)).filter(F.col("city_rank_lowest") == 1).show()

+---------+-----------+--------------+---------+-------+----------------+
|doctor_id|doctor_name|specialization|     city|revenue|city_rank_lowest|
+---------+-----------+--------------+---------+-------+----------------+
|      ...|       NULL|          NULL|     NULL|      0|               1|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|   5000|               1|
+---------+-----------+--------------+---------+-------+----------------+



In [0]:
df_rev_base.withColumn("Leaderboard_Pos", F.dense_rank().over(win_global_rev)).orderBy("Leaderboard_Pos").show()

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+---------------+
|doctor_id|doctor_name|specialization|     city|revenue|Leaderboard_Pos|
+---------+-----------+--------------+---------+-------+---------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|   5000|              1|
|      ...|       NULL|          NULL|     NULL|      0|              2|
+---------+-----------+--------------+---------+-------+---------------+



In [0]:
df_docs.createOrReplaceTempView("sql_doctors")
df_visits_clean.createOrReplaceTempView("sql_visits")
df_json.createOrReplaceTempView("sql_hospitals")

In [0]:
spark.sql("SELECT * FROM sql_doctors").show()

+---------+-----------+--------------+---------+----------------+----------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_fee|
+---------+-----------+--------------+---------+----------------+----------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|            1500|
|      ...|       NULL|          NULL|     NULL|            NULL|            NULL|
+---------+-----------+--------------+---------+----------------+----------------+



In [0]:
spark.sql("SELECT specialization, COUNT(*) FROM sql_doctors GROUP BY specialization").show()

+--------------+--------+
|specialization|count(1)|
+--------------+--------+
|          NULL|       1|
|    Cardiology|       1|
+--------------+--------+



In [0]:
spark.sql("SELECT city, COUNT(*) FROM sql_doctors GROUP BY city").show()

+---------+--------+
|     city|count(1)|
+---------+--------+
|     NULL|       1|
|Hyderabad|       1|
+---------+--------+



In [0]:
spark.sql("""SELECT d.doctor_name, SUM(v.bill_amount) as revenue 
             FROM sql_doctors d LEFT JOIN sql_visits v ON d.doctor_id = v.doctor_id 
             GROUP BY d.doctor_name""").show()

+-----------+-------+
|doctor_name|revenue|
+-----------+-------+
|       NULL|   NULL|
| Dr. Ramesh|   5000|
+-----------+-------+



In [0]:
spark.sql("""SELECT d.specialization, SUM(v.bill_amount) as revenue 
             FROM sql_doctors d JOIN sql_visits v ON d.doctor_id = v.doctor_id 
             GROUP BY d.specialization""").show()

+--------------+-------+
|specialization|revenue|
+--------------+-------+
|    Cardiology|   5000|
+--------------+-------+



In [0]:
spark.sql("""SELECT d.doctor_name, COALESCE(SUM(v.bill_amount), 0) as total_revenue 
             FROM sql_doctors d LEFT JOIN sql_visits v ON d.doctor_id = v.doctor_id 
             GROUP BY d.doctor_name ORDER BY total_revenue DESC LIMIT 5""").show()

+-----------+-------------+
|doctor_name|total_revenue|
+-----------+-------------+
| Dr. Ramesh|         5000|
|       NULL|            0|
+-----------+-------------+



In [0]:
spark.sql("SELECT * FROM sql_visits WHERE payment_status = 'Pending'").show()

+--------+------------+---------+----------+---------+-----------+--------------+
|visit_id|patient_name|doctor_id|visit_date|diagnosis|bill_amount|payment_status|
+--------+------------+---------+----------+---------+-----------+--------------+
+--------+------------+---------+----------+---------+-----------+--------------+



In [0]:
spark.sql("SELECT hospital_name FROM sql_hospitals WHERE array_contains(services, 'Cardiology')").show()
spark.sql("SELECT hospital_name FROM sql_hospitals WHERE array_contains(services, 'Neurology')").show()

+----------------+
|   hospital_name|
+----------------+
| Apollo Hospital|
|Yashoda Hospital|
+----------------+

+---------------+
|  hospital_name|
+---------------+
|Apollo Hospital|
|  Care Hospital|
+---------------+



In [0]:
spark.sql("SELECT hospital_name FROM sql_hospitals WHERE contact.phone IS NULL OR contact.email IS NULL").show()

+----------------+
|   hospital_name|
+----------------+
|Yashoda Hospital|
|   Care Hospital|
+----------------+



In [0]:
spark.sql("SELECT AVG(consultation_fee) as system_avg_fee FROM sql_doctors").show()

+--------------+
|system_avg_fee|
+--------------+
|        1500.0|
+--------------+



In [0]:
spark.sql("""
    SELECT d.doctor_id, d.doctor_name, COUNT(v.visit_id) as patient_count, COALESCE(SUM(v.bill_amount),0) as revenue
    FROM sql_doctors d
    LEFT JOIN sql_visits v ON d.doctor_id = v.doctor_id
    GROUP BY d.doctor_id, d.doctor_name
    ORDER BY revenue DESC
""").show()

+---------+-----------+-------------+-------+
|doctor_id|doctor_name|patient_count|revenue|
+---------+-----------+-------------+-------+
|     D101| Dr. Ramesh|            1|   5000|
|      ...|       NULL|            0|      0|
+---------+-----------+-------------+-------+



In [0]:
raw_docs = spark.read.option("header","true").option("inferSchema","true").csv("dbfs:/tmp/doctors.csv")
raw_visits = spark.read.option("header","true").option("inferSchema","true").csv("dbfs:/tmp/visits.csv")
raw_hospitals = spark.read.option("multiline","true").json("dbfs:/tmp/hospital_config.json")

In [0]:
cleaned_visits = raw_visits.fillna({"bill_amount": 0})
flat_hospitals = raw_hospitals.withColumn("phone", F.col("contact.phone")) \
                              .withColumn("email", F.col("contact.email")) \
                              .withColumn("service_line", F.explode(F.col("services"))) \
                              .drop("contact", "services")

In [0]:
silver_clinical_encounters = raw_docs.join(cleaned_visits, "doctor_id", "inner")

In [0]:
silver_financial_enriched = silver_clinical_encounters.withColumn("tax_amount", F.col("bill_amount") * 0.05) \
                                                      .withColumn("gross_revenue", F.col("bill_amount") + F.col("tax_amount"))

In [0]:
win_analytics = Window.partitionBy("specialization").orderBy(F.desc("bill_amount"))
silver_final = silver_financial_enriched.withColumn("spec_visit_rank", F.dense_rank().over(win_analytics))

In [0]:
gold_spec_summary = silver_final.groupBy("specialization").agg(
    F.count("visit_id").alias("total_visits"),
    F.sum("bill_amount").alias("net_revenue"),
    F.sum("tax_amount").alias("tax_collected")
)

In [0]:
silver_final.write.mode("overwrite").parquet("/tmp/silver/clinical_encounters")
gold_spec_summary.write.mode("overwrite").parquet("/tmp/gold/specialization_summary")

In [0]:
read_gold_summary = spark.read.parquet("/tmp/gold/specialization_summary")

In [0]:
dashboard_dataset = flat_hospitals.join(read_gold_summary, flat_hospitals.service_line == read_gold_summary.specialization, "left")
display(dashboard_dataset)